# 03. Lattice dynamics and annealing

这个 notebook 整合 lattice/anneal 相关脚本：

- `run_route2_lattice_cell.py`：Route-2 3x3 cell 的 single Rydberg pi/2 pulse。
- `run_3level_lattice.py`：3-level `01r` lattice sweep，制备 checkerboard Rydberg pattern。
- `run_15x15_anneal.py`：通用 TN backend anneal runner。
- `run_anneal_batch.py`：main.tex Anneal I/II batch runner。

这条线和 CZ/local-addressing 分开：这里关心的是多体 lattice dynamics、observable recording 和 tensor-network backend 配置。


## 0. Notebook setup

这个 block 准备 repo path 和 lattice/anneal API。预期结果是 exact 小系统可以直接跑；TN 大系统默认 dry-run，避免在没有可选 backend 的环境中失败。


In [ ]:
from pathlib import Path
import os
import sys
import json
from itertools import product

os.environ.setdefault("JAX_PLATFORMS", "cpu")

HERE = Path.cwd().resolve()
if HERE.name == "notebooks":
    REPO_ROOT = HERE.parents[1]
elif HERE.name == "scripts":
    REPO_ROOT = HERE.parent
else:
    REPO_ROOT = HERE
os.chdir(REPO_ROOT)

for extra in (REPO_ROOT / "src", REPO_ROOT / "scripts"):
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from ryd_gate import RydbergSystem
from ryd_gate.backends.exact import simulate
from ryd_gate.core.level_structures import InteractionSpec
from ryd_gate.lattice import make_square_lattice, plot_spatial_rydberg
from ryd_gate.protocols.digital_analog import DigitalAnalogProtocol, Segment
from ryd_gate.analysis.lattice_observables import measure_rydberg_occupation, precompute_trit_masks, staggered_magnetization
from ryd_gate.backends.tn_common import create_tn_lattice_spec, simulate_tn
from ryd_gate.protocols.lattice_dynamics import TFIMAnnealProtocol

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25})
print(f"Repo root: {REPO_ROOT}")


## 1. Route-2 lattice cell: single Rydberg pi/2 pulse

这个 block 对应 `run_route2_lattice_cell.py`。从 `|111...>` 出发，对 3x3 lattice 施加 resonant Rydberg `π/2` pulse。预期结果是每个 site 的 `|1>` 和 `|r>` population 接近 1/2，并且 final state 与 `(|1> - i|r>)/sqrt(2)` 的 product state 有高 overlap。物理意义是验证 route-2 中用于生成 local superposition 的基本 pulse cell。


In [ ]:
OMEGA_R = 2 * np.pi * 200e6
Lx, Ly = 3, 3
spacing = 10.0
geom = make_square_lattice(Lx, Ly, spacing_um=spacing)
N = geom.N

t_pi2_R = np.pi / (2 * OMEGA_R)
proto_direct = DigitalAnalogProtocol([Segment(duration=t_pi2_R, omega_R=OMEGA_R)], n_steps=200)
system_route2 = RydbergSystem.from_lattice(
    geom,
    level_structure="01r",
    interaction=InteractionSpec(),
    protocol=proto_direct,
)
psi0 = system_route2.product_state(["1"] * N)
result = simulate(system_route2, [], psi0, t_eval=True)
times = np.concatenate([[0.0], result.times])
if isinstance(result.states, np.ndarray) and result.states.ndim == 2:
    states = [psi0] + [result.states[:, k] for k in range(result.states.shape[1])]
else:
    states = [psi0, *result.states]

levels = tuple(system_route2.basis.local_levels)
pops_site = {
    i: {lvl: np.array([system_route2.expectation(f"n_{lvl}_{i}", psi) for psi in states]) for lvl in levels}
    for i in range(N)
}
psi_final = states[-1]
psi_plus = sum(
    (-1j) ** sum(c == "r" for c in cfg) * system_route2.product_state(list(cfg))
    for cfg in product(["1", "r"], repeat=N)
) / (2 ** (N / 2))
fidelity = np.abs(np.vdot(psi_plus, psi_final)) ** 2
print(f"N={N}, fidelity to product (|1>-i|r>)/sqrt(2): {fidelity:.6f}")

colors = {"0": "tab:gray", "1": "tab:blue", "r": "tab:red"}
fig, axes = plt.subplots(Ly, Lx, figsize=(3.2 * Lx, 2.7 * Ly), sharex=True, sharey=True)
for i in range(N):
    ix, iy = i // Ly, i % Ly
    ax = axes[iy, ix]
    for lvl in ("0", "1", "r"):
        ax.plot(times / t_pi2_R, pops_site[i][lvl], color=colors[lvl], lw=2, label=rf"$n_{lvl}$")
    ax.axhline(0.5, color="k", ls=":", lw=1)
    ax.set_title(f"site ({ix}, {iy})")
    if i == 0:
        ax.legend(loc="center right", fontsize=8)
for ax in axes[-1, :]:
    ax.set_xlabel(r"time / $t_{\pi/2}$")
for ax in axes[:, 0]:
    ax.set_ylabel("population")
fig.suptitle(r"Route 2: single $\pi/2$ Rydberg pulse on a 3x3 lattice")
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(4, 1, figsize=(10, 6), sharex=True)
fields = [("omega_R", r"$\Omega_R$"), ("omega_hf", r"$\Omega_{hf}$"), ("delta_R", r"$\Delta_R$"), ("delta_hf", r"$\Delta_{hf}$")]
t_edges = [0.0]
for seg in proto_direct.segments:
    t_edges.append(t_edges[-1] + seg.duration)
for ax, (field, label) in zip(axes, fields):
    t_pts, y_pts = [], []
    for j, seg in enumerate(proto_direct.segments):
        value = getattr(seg, field)
        t_pts.extend([t_edges[j], t_edges[j + 1]])
        y_pts.extend([value, value])
    ax.step(t_pts, y_pts, where="post", lw=2)
    ax.axhline(0, color="k", ls=":", lw=0.8)
    ax.set_ylabel(label)
axes[-1].set_xlabel("time (s)")
fig.suptitle("Digital-analog protocol schedule")
fig.tight_layout()
plt.show()


## 2. Exact 3-level lattice sweep to a checkerboard pattern

这个 block 对应 `run_3level_lattice.py`。从 `|111...>` 出发，扫 `|1> ↔ |r>` detuning，在 Rydberg interaction 下制备 checkerboard-like ordered state。预期结果是 final Rydberg occupation 在两个 sublattice 上交替，staggered magnetization 非零。物理意义是展示小 lattice exact dynamics 如何捕捉 Rydberg blockade 诱导的空间有序。


In [ ]:
Lx, Ly = 3, 3
geom_cb = make_square_lattice(Lx, Ly, spacing_um=5.0)
n_steps = 300
delta_start = -2 * np.pi * 40e6
delta_end = 2 * np.pi * 40e6
t_sweep = 1.5e-6
omega_R = 2 * np.pi * 5e6
dt = t_sweep / n_steps
segments = [
    Segment(duration=dt, omega_R=omega_R, delta_R=delta_start + (delta_end - delta_start) * (k + 0.5) / n_steps)
    for k in range(n_steps)
]
protocol_cb = DigitalAnalogProtocol(segments, n_steps=n_steps)
system_cb = RydbergSystem.from_lattice(
    geom_cb,
    "01r",
    interaction=InteractionSpec(mode="all"),
    protocol=protocol_cb,
)
psi0_cb = system_cb.product_state(["1"] * geom_cb.N)
print(f"Exact Hilbert dimension: 3^{geom_cb.N} = {3 ** geom_cb.N}")
result_cb = simulate(system_cb, [], psi0_cb)

masks = precompute_trit_masks(geom_cb.N)
occ_final = measure_rydberg_occupation(result_cb.psi_final, masks)
ms = staggered_magnetization(occ_final, geom_cb.sublattice)
print(f"Final staggered magnetization m_s = {ms:.4f}")

fig = plot_spatial_rydberg(
    geom_cb.coords,
    occ_final,
    geom_cb.sublattice,
    title=f"3x3 checkerboard after adiabatic sweep (m_s={ms:.2f})",
)
plt.show()

if result_cb.states is not None:
    occ_traj = measure_rydberg_occupation(result_cb.states, masks)
    fig, ax = plt.subplots(figsize=(8, 4.8))
    for i in range(geom_cb.N):
        color = "tab:red" if geom_cb.sublattice[i] > 0 else "tab:blue"
        ax.plot(result_cb.times * 1e6, occ_traj[i], color=color, alpha=0.7, lw=1.2)
    ax.set_xlabel("time (us)")
    ax.set_ylabel(r"$P_r$ per site")
    ax.set_title("Rydberg occupation during checkerboard sweep")
    fig.tight_layout()
    plt.show()


## 3. Tensor-network anneal smoke test

这个 block 对应 `run_15x15_anneal.py`。默认只构造 4x4 configuration 并 dry-run 打印，因为 TN backend 可能依赖可选包、CUDA 或 Julia。把 `RUN_TN_SMOKE=True` 后，会通过 `simulate_tn` 运行并画 observable traces。默认 preset 使用 `backend="mps_gpu"`，即 YASTN MPS-TDVP；也可以切到 `backend="2dtn"` 并设置 `engine_package="yastn"` 或 `"quimb"` 来测试 Python 2DTN concrete adapters，或切到 `backend="nqs"`、`engine_package="netket"` 运行 NQS-tVMC。预期结果：dry-run 给出 backend、protocol、observables 和 lattice metadata；如果运行成功，得到 `sigma_z`、`czz_centerline` 等时间序列。物理意义是把 exact 小系统推广到可处理 15x15 的统一 backend API。


In [ ]:
RUN_TN_SMOKE = False

Lx, Ly = 4, 4
backend = "mps_gpu"  # YASTN MPS-TDVP; set use_cuda=True on a CUDA/PyTorch machine
spec = create_tn_lattice_spec(
    Lx,
    Ly,
    V_nn=24.0,
    Omega=2.0,
    level_structure="1r",
    interaction_mode="nn",
    ordering="snake",
)
protocol = TFIMAnnealProtocol(
    hx_peak=1.0,
    hx_initial=0.0,
    hx_final=0.0,
    hz_initial=4.0,
    hz_final=-4.0,
    t_rise=2.0,
    t_sweep=8.0,
    t_fall=2.0,
)
t_eval = np.linspace(0.0, protocol.t_gate, 13)
observables = ["sigma_z", "czz_centerline"]
backend_options = {"chi_max": 256, "dt": 0.1, "svd_min": 1e-10, "use_cuda": False}
# Alternative 2D-TN concrete adapters:
# backend = "2dtn"
# backend_options = {"engine_package": "yastn", "chi_max": 64, "dt": 0.1, "svd_min": 1e-10, "use_cuda": False}
# backend_options = {"engine_package": "quimb", "chi_max": 64, "dt": 0.1, "svd_min": 1e-10, "use_cuda": False}
# Alternative NQS-tVMC concrete adapter:
# backend = "nqs"
# backend_options = {"engine_package": "netket", "dt": 0.02, "sampler": "metropolis", "n_samples": 2048, "n_chains": 32, "ansatz": "rbm", "alpha": 1, "use_cuda": False}

config = {
    "Lx": spec.Lx,
    "Ly": spec.Ly,
    "N": spec.N,
    "backend": backend,
    "protocol": {"hx_peak": 1.0, "hz_initial": 4.0, "hz_final": -4.0, "t_gate": protocol.t_gate},
    "observables": observables,
    "t_eval_steps": len(t_eval),
    "backend_options": backend_options,
}
print(json.dumps(config, indent=2, sort_keys=True))

if RUN_TN_SMOKE:
    result_tn = simulate_tn(
        spec,
        protocol,
        [],
        initial_state="all_ground",
        backend=backend,
        t_eval=t_eval,
        observables=observables,
        backend_options=backend_options,
    )
    obs = result_tn.metadata.get("obs") or {}
    for name, values in obs.items():
        arr = np.asarray(values)
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(result_tn.times, arr.reshape(len(result_tn.times), -1).mean(axis=1), "o-")
        ax.set_xlabel("time")
        ax.set_ylabel(name)
        ax.set_title(f"TN observable: {name}")
        fig.tight_layout()
        plt.show()
else:
    print("Dry-run only. Set RUN_TN_SMOKE=True to invoke the selected backend.")


## 4. Production 15x15 configuration

这个 block 对应 `run_15x15_anneal.py --production-15x15`。我们只展示 production 配置，不默认运行。预期结果是清楚看到 15x15 lattice、backend、bond dimension 和 observable 设置。物理意义是给大系统 benchmark 一个可复现的参数入口，同时避免 notebook 默认执行昂贵任务。


In [ ]:
production = {
    "notebook_equivalent": "Set Lx=Ly=15 in the TN configuration cell and enable RUN_TN_SMOKE with the desired backend/output handling.",
    "Lx": 15,
    "Ly": 15,
    "backend_choices": ["tenpy", "mps", "mps_gpu", "itensors", "gputn", "gputtn", "2dtn", "ttn", "nqs"],
    "default_observables": ["sigma_z", "czz_centerline"],
    "recommended_note": "Run outside the notebook when using production lattice sizes or optional Julia/GPU/NetKet backends.",
}
print(json.dumps(production, indent=2))


## 5. Anneal I/II batch results

这个 block 对应 `run_anneal_batch.py`。默认扫描 `scripts/results/anneal_batch/*.npz`，如果已有 batch 结果就直接画；如果没有，只显示等价命令。预期结果是把不同 schedule/sweep value/backend 的 observable 文件集中检查。物理意义是复现 main.tex 中 Anneal I/II benchmark 的入口。


In [ ]:
batch_dir = REPO_ROOT / "scripts" / "results" / "anneal_batch"
files = sorted(batch_dir.glob("*.npz")) if batch_dir.exists() else []

if not files:
    print("No cached batch .npz files found.")
    print("Example command:")
    print("Use the Anneal I/II configuration in this notebook, or build the same run_config dictionary and call simulate_tn directly.")
else:
    print(f"Found {len(files)} batch file(s).")
    for path in files[:6]:
        data = np.load(path, allow_pickle=False)
        print(f"\n{path.name}: arrays = {list(data.files)}")
        if "times" not in data.files:
            continue
        times = data["times"]
        obs_keys = [k for k in data.files if k.startswith("obs_")]
        for key in obs_keys:
            arr = np.asarray(data[key])
            fig, ax = plt.subplots(figsize=(7, 4))
            ax.plot(times, arr.reshape(len(times), -1).mean(axis=1), "o-")
            ax.set_xlabel("time")
            ax.set_ylabel(key.replace("obs_", ""))
            ax.set_title(path.name)
            fig.tight_layout()
            plt.show()
